In [ ]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 31.5 MB/s eta 0:00:00


## Detech Person and Racket using yolov8x.pt

In [ ]:
from ultralytics import YOLO
import cv2
import pickle
import csv

# Load YOLOv8 model
model = YOLO("yolov8x.pt")

PERSON = 0
RACKET = 38

video_path = "/content/input.mp4"
cap = cv2.VideoCapture(video_path)

# Fix FPS issue (common in Colab)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS)

if fps is None or fps == 0:
    fps = 30

# Output video (Colab-safe path)
out = cv2.VideoWriter(
    "/content/full_output.mp4",
    cv2.VideoWriter_fourcc(*"mp4v"),
    fps,
    (width, height)
)

# Storage
all_data = []

# CSV file
csv_file = open("/content/full_detections.csv", "w", newline="")
csv_writer = csv.writer(csv_file)

csv_writer.writerow([
    "frame",
    "track_id",
    "label",
    "x1",
    "y1",
    "x2",
    "y2",
    "confidence"
])

frame_id = 0

while True:
    ret, frame = cap.read()
    if not ret:
        break

    # Tracking
    results = model.track(
        frame,
        persist=True,
        tracker="bytetrack.yaml",
        classes=[PERSON, RACKET],
        imgsz=1280,
        conf=0.15
    )

    frame_data = []
    r = results[0]

    # Safety check (important in Colab + tracking)
    if r.boxes is None or r.boxes.id is None:
        all_data.append(frame_data)
        frame_id += 1
        continue

    boxes = r.boxes.xyxy.cpu().numpy()
    classes = r.boxes.cls.cpu().numpy().astype(int)
    confs = r.boxes.conf.cpu().numpy()
    track_ids = r.boxes.id.cpu().numpy().astype(int)

    for box, cls, conf, track_id in zip(boxes, classes, confs, track_ids):

        x1, y1, x2, y2 = map(int, box)

        label = "person" if cls == PERSON else "racket"

        data = {
            "frame": frame_id,
            "track_id": int(track_id),
            "label": label,
            "bbox": [x1, y1, x2, y2],
            "confidence": float(conf)
        }

        frame_data.append(data)

        csv_writer.writerow([
            frame_id,
            int(track_id),
            label,
            x1,
            y1,
            x2,
            y2,
            float(conf)
        ])

        # Draw boxes
        color = (0, 255, 0) if cls == PERSON else (255, 0, 0)

        cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)

        cv2.putText(
            frame,
            f"{label} ID:{track_id}",
            (x1, y1 - 10),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.7,
            color,
            2
        )

    all_data.append(frame_data)

    out.write(frame)

    frame_id += 1

cap.release()
out.release()
csv_file.close()

# Save PKL
with open("/content/full_detections.pkl", "wb") as f:
    pickle.dump(all_data, f)

print("saved -> /content/detections.pkl")
print("saved -> /content/detections.csv")
print("saved -> /content/new_output.mp4")

Streaming output truncated to the last 5000 lines.
0: 736x1280 1 tennis-ball, 12.2ms
Speed: 9.9ms preprocess, 12.2ms inference, 1.5ms postprocess per image at shape (1, 3, 736, 1280)

0: 736x1280 1 tennis-ball, 12.1ms
Speed: 9.2ms preprocess, 12.1ms inference, 1.5ms postprocess per image at shape (1, 3, 736, 1280)

0: 736x1280 1 tennis-ball, 12.1ms
Speed: 10.2ms preprocess, 12.1ms inference, 1.5ms postprocess per image at shape (1, 3, 736, 1280)

0: 736x1280 1 tennis-ball, 12.1ms
Speed: 11.1ms preprocess, 12.1ms inference, 1.5ms postprocess per image at shape (1, 3, 736, 1280)

0: 736x1280 1 tennis-ball, 12.1ms
Speed: 9.9ms preprocess, 12.1ms inference, 1.5ms postprocess per image at shape (1, 3, 736, 1280)

0: 736x1280 1 tennis-ball, 12.2ms
Speed: 11.6ms preprocess, 12.2ms inference, 2.4ms postprocess per image at shape (1, 3, 736, 1280)

0: 736x1280 1 tennis-ball, 12.1ms
Speed: 8.9ms preprocess, 12.1ms inference, 1.4ms postprocess per image at shape (1, 3, 736, 1280)

0: 736x1280 1 t

## Detect ball using https://huggingface.co/RJTPP/tennis-ball-detection/blob/main/best.pt


In [ ]:
from ultralytics import YOLO
import cv2
import pickle
import csv

model = YOLO("tennis_ball.pt")

video_path = "/content/input.mp4"
cap = cv2.VideoCapture(video_path)

width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS) or 30

out = cv2.VideoWriter(
    "/content/ball_output.mp4",
    cv2.VideoWriter_fourcc(*"mp4v"),
    fps,
    (width, height)
)

csv_file = open("/content/ball_detections.csv", "w", newline="")
csv_writer = csv.writer(csv_file)

csv_writer.writerow(["frame", "x1", "y1", "x2", "y2", "conf"])

all_data = []
frame_id = 0

while True:
    ret, frame = cap.read()
    if not ret:
        break

    results = model.predict(
        frame,
        imgsz=1920,
        conf=0.10
    )

    r = results[0]
    frame_data = []

    if r.boxes is not None:
        boxes = r.boxes.xyxy.cpu().numpy()
        confs = r.boxes.conf.cpu().numpy()

        for box, conf in zip(boxes, confs):
            x1, y1, x2, y2 = map(int, box)

            frame_data.append({
                "frame": frame_id,
                "bbox": [x1, y1, x2, y2],
                "confidence": float(conf)
            })

            csv_writer.writerow([frame_id, x1, y1, x2, y2, float(conf)])

            cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 255), 2)

    all_data.append(frame_data)
    out.write(frame)

    frame_id += 1

cap.release()
out.release()
csv_file.close()

with open("/content/ball_detections.pkl", "wb") as f:
    pickle.dump(all_data, f)

print("Done")

Streaming output truncated to the last 5000 lines.
Speed: 9.8ms preprocess, 22.4ms inference, 1.3ms postprocess per image at shape (1, 3, 1088, 1920)

0: 1088x1920 3 tennis-balls, 22.4ms
Speed: 9.9ms preprocess, 22.4ms inference, 1.3ms postprocess per image at shape (1, 3, 1088, 1920)

0: 1088x1920 2 tennis-balls, 22.3ms
Speed: 9.7ms preprocess, 22.3ms inference, 1.4ms postprocess per image at shape (1, 3, 1088, 1920)

0: 1088x1920 4 tennis-balls, 22.4ms
Speed: 10.6ms preprocess, 22.4ms inference, 2.1ms postprocess per image at shape (1, 3, 1088, 1920)

0: 1088x1920 2 tennis-balls, 22.3ms
Speed: 12.5ms preprocess, 22.3ms inference, 1.3ms postprocess per image at shape (1, 3, 1088, 1920)

0: 1088x1920 2 tennis-balls, 22.3ms
Speed: 10.4ms preprocess, 22.3ms inference, 1.3ms postprocess per image at shape (1, 3, 1088, 1920)

0: 1088x1920 2 tennis-balls, 22.3ms
Speed: 11.0ms preprocess, 22.3ms inference, 1.5ms postprocess per image at shape (1, 3, 1088, 1920)

0: 1088x1920 2 tennis-balls, 